* ### Analyse 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import pathlib
import os
from sklearn.preprocessing import StandardScaler,MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
import xgboost
import lime
import shape

: 

In [ ]:
os.chdir(path="/home/donerick/Challenge 30 Days ML")

In [ ]:
data_air = pd.read_csv(pathlib.Path("data/raw/hourly_quality_air_data.csv"))

data_meteo = pd.read_csv(pathlib.Path("data/raw/hourly_meteo_data.csv"))

In [ ]:
data_air.head()

In [ ]:
data_air.shape

In [ ]:
data_meteo.head()

In [ ]:
data_meteo.dtypes

In [ ]:
def merge_sources(meteo: pd.DataFrame, air: pd.DataFrame) -> pd.DataFrame:
    """
    Merge météo + qualité d'air sur la colonne 'date' (timestamp exact).
    Les deux APIs Open-Meteo produisent des timestamps horaires alignés —
    un inner join suffit..
    """
    meteo['date'] = pd.to_datetime(meteo["date"], utc=True)
    air['date'] = pd.to_datetime(air['date'], utc=True)
 
    merged = pd.merge(
        meteo,
        air,
        on='date',
        how="inner"
        ).reset_index()
 
 
    before = len(merged)
    merged = merged.dropna(subset=["pm10", "pm2_5", "temperature_2m"])
    after  = len(merged)
    if before != after:
        print(f"Lignes supprimées après merge (NaN critiques) : {before - after}")
 
    print(f"Après merge : {len(merged)} lignes, {merged.shape[1]} colonnes")
    return merged
 
 

In [ ]:
merge_sources(data_air, data_meteo)

In [ ]:
df_merged = merge_sources(data_air, data_meteo)

df_merged.describe()

In [ ]:
df_merged.isnull().sum()

In [ ]:
df_merged= df_merged.drop(columns=['index', "formaldehyde"], axis=1)
df_merged.sample(10)

In [ ]:
df_merged.to_csv('data/processed/df.csv',index=False)

In [ ]:
corr = df_merged.corr(numeric_only=True)["pm2_5"].sort_values(ascending=False)
print(corr)

#### Preparation des données 

In [ ]:
import numpy as np
def prepare_data(df):

    # extract features of date
    df = df.copy()
    df['hour'] = df['date'].dt.hour
    df['dayofweek'] = df['date'].dt.dayofweek

    # encodage cyclique sur 24h
    df.loc[:, 'hour_sin'] = np.sin(2*np.pi*df['hour']/24)
    df.loc[:, 'hour_cos'] = np.cos(2*np.pi*df['hour']/24)

    # moyenne mobile sur les 24 h
    for lag in [1, 3, 6, 12, 24]:
        df.loc[:, f'aqi_lag_{lag}'] = df['european_aqi'].shift(lag)

    # moyenne mobile de pm2.5 sur les 24h
    for lag in [1, 3, 6, 12, 24]:
        df.loc[:, f'pm2_5_lag_{lag}'] = df['pm2_5'].shift(lag)

    # rolling mean
    df.loc[:, 'rolling_aqi_mean'] = df['european_aqi'].shift(1).rolling(6).mean()


    # target features

    df.loc[:, 'PM2.5'] = df['pm2_5'].shift(-1)
    df.loc[:, "AQI"] = df['european_aqi'].shift(-1)

    df = df.dropna()

    return df

In [ ]:
df = prepare_data(df=df_merged)
df.head(5)

In [ ]:
from sklearn.multioutput import MultiOutputRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split



In [ ]:
def train_model(df):

    df_feat = prepare_data(df)

    features = [col for col in df_feat.columns 
                if col not in ["date", "AQI", "PM2.5", "pm2_5", "european_aqi"]]

    X = df_feat[features]
    y = df_feat[["AQI", "PM2.5"]]


    X_train, X_test, y_train, y_test = train_test_split(
        X, y, shuffle=False, test_size=0.3
    )

    base_model = XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.03,
        gamma=0.1,
        eval_metric="rmse",
        reg_alpha=0.1
    )

    model = MultiOutputRegressor(base_model)

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)



    return model, features, X_test, y_test, y_pred

In [ ]:
model, features, X_test, y_test, y_pred = train_model(df=df)
mae_aqi = mean_absolute_error(y_test["AQI"], y_pred[:,0])
mae_pm  = mean_absolute_error(y_test["PM2.5"], y_pred[:,1])

print("MAE AQI:", mae_aqi)
print("MAE PM2.5:", mae_pm)

In [ ]:
model

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(y_test["PM2.5"], y_pred[:,1])
plt.xlabel("Réel")
plt.ylabel("Prédit")
plt.title("PM2.5 prediction")

In [ ]:
import pandas as pd

def get_feature_importance(model, features):

    # AQI
    aqi_importance = pd.Series(
        model.estimators_[0].feature_importances_,
        index=features
    ).sort_values(ascending=False)

    # PM2.5
    pm25_importance = pd.Series(
        model.estimators_[1].feature_importances_,
        index=features
    ).sort_values(ascending=False)

    return aqi_importance, pm25_importance

In [ ]:
import plotly.express as px

def plot_importance(importance, title):

    df_imp = importance.reset_index()
    df_imp.columns = ["feature", "importance"]

    fig = px.bar(
        df_imp.head(15),
        x="importance",
        y="feature",
        orientation="h",
        title=title
    )

    fig.update_layout(yaxis=dict(autorange="reversed"))

    return fig
aqi_imp, pm_imp = get_feature_importance(model, features)
fig_aqi = plot_importance(aqi_imp, "Importance des variables — AQI")
fig_pm  = plot_importance(pm_imp,  "Importance des variables — PM2.5")

fig_aqi.show()
fig_pm.show()

In [ ]:
def create_features(data):

    df = data.copy()

    # drop propre
    dropped_columns = [
        "european_aqi_pm2_5",
        "carbon_monoxide",
        "methane",
        "wind_direction_10m",
        "soil_temperature_0_to_7cm",
        "soil_moisture_0_to_7cm",
        "precipitation",
        "rain",
        "pressure_msl",
        "surface_pressure",
    ]

    df = df.drop(columns=[col for col in dropped_columns if col in df.columns])

    # date
    df["date"] = pd.to_datetime(df["date"], utc=True)
    df["hour"] = df["date"].dt.hour

    # cyclique
    df.loc[:, "hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
    df.loc[:, "hour_cos"] = np.cos(2*np.pi*df["hour"]/24)

    # lags AQI
    for lag in [1, 3, 6, 24]:
        df.loc[:, f"aqi_lag_{lag}"] = df["european_aqi"].shift(lag)

    # lags PM2.5
    for lag in [1, 3, 24]:
        df.loc[:, f"pm2_5_lag_{lag}"] = df["pm2_5"].shift(lag)

    # targets
    df["PM25"] = df["pm2_5"].shift(-1)
    df["AQI"]  = df["european_aqi"].shift(-1)

    # drop inutiles
    df = df.drop(columns=["hour"])

    df = df.dropna()

    return df

In [ ]:
create_features(data=df_merged)